# 05 Final Load Prep

Use this notebook to compute final KPIs, prepare the Tableau-ready dataset, and export the exact file used in the dashboard.

In [ ]:
#Environment Setup & Data Import

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#Point to the processed folder
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'RTA_cleaned.csv'

try:
    df = pd.read_csv(PROCESSED_PATH)
    print(f"✓ Cleaned dataset loaded for final prep. Shape: {df.shape}")
except FileNotFoundError:
    print("CRITICAL ERROR: Cleaned dataset not found. Execute 02_cleaning.ipynb first.")
    raise

✓ Cleaned dataset loaded for final prep. Shape: (12316, 35)


In [ ]:
#Pre-computing Strategic KPIs

print("="*80)
print("TRAFFIC AUTHORITY KPI COMPUTATION")
print("="*80)

# 1. Total Volume
total_accidents = len(df)
print(f"Total Accidents Recorded: {total_accidents:,}")

# 2. Severe Incident Volume
severe_accidents = len(df[df['Accident_severity'].isin(['Serious Injury', 'Fatal injury'])])
severe_rate = (severe_accidents / total_accidents) * 100
print(f"Severe Incidents: {severe_accidents:,} ({severe_rate:.2f}% of total)")

# 3. High-Risk Environment Impact
if 'Is_Adverse_Condition' in df.columns:
    adverse_incidents = len(df[df['Is_Adverse_Condition'] == 'Yes'])
    adverse_rate = (adverse_incidents / total_accidents) * 100
    print(f"Incidents in Adverse Conditions: {adverse_incidents:,} ({adverse_rate:.2f}% of total)")

# 4. Total Casualties
if 'Number_of_casualties' in df.columns:
    total_casualties = df['Number_of_casualties'].sum()
    print(f"Total Casualties: {total_casualties:,}")

TRAFFIC AUTHORITY KPI COMPUTATION
Total Accidents Recorded: 12,316
Severe Incidents: 1,743 (14.15% of total)
Incidents in Adverse Conditions: 3,441 (27.94% of total)
Total Casualties: 19,067


In [ ]:
# CELL 3: Tableau Optimization & Type Flattening

print("\nOptimizing data types for Tableau Public compatibility...")

# Convert all category types back to clean strings to prevent Tableau import errors
categorical_cols = df.select_dtypes(include=['category']).columns
for col in categorical_cols:
    df[col] = df[col].astype(str)

# Ensure numeric columns are explicitly int/float
numeric_cols = ['Number_of_vehicles_involved', 'Number_of_casualties', 'Severity_score']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print("All data types flattened and verified.")


Optimizing data types for Tableau Public compatibility...
✓ All data types flattened and verified.


In [ ]:
#Master Tableau Export

TABLEAU_EXPORT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'rta_tableau_master.csv'

# Export the flattened, enriched dataset
df.to_csv(TABLEAU_EXPORT_PATH, index=False)

print("="*80)
print("FINAL PIPELINE COMPLETE")
print("="*80)
print(f"Master Tableau file saved to: {TABLEAU_EXPORT_PATH}")
print(f"Final feature count ready for dashboarding: {len(df.columns)}")
print("Next Step: Open Tableau Public and connect to 'rta_tableau_master.csv'.")

FINAL PIPELINE COMPLETE
✓ Master Tableau file saved to: /Users/rohankumar/Desktop/DVA-ModelCraft-Retail/data/processed/rta_tableau_master.csv
✓ Final feature count ready for dashboarding: 35
→ Next Step: Open Tableau Public and connect to 'rta_tableau_master.csv'.
